In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")
dbutils.widgets.text("null_high_severity_pct", "10")
dbutils.widgets.text("dq_sample_limit", "20")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
null_high_severity_pct = float(dbutils.widgets.get("null_high_severity_pct"))
dq_sample_limit = int(dbutils.widgets.get("dq_sample_limit"))

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

## Step 3 - Ingest CSV files into Bronze Layer

In [ ]:
from actuarial_claim_pipeline.bronze import filter_csv_files, ingest_volume_csvs

# Preview CSV discovery (ingest_volume_csvs raises if empty)
csv_files = filter_csv_files(dbutils.fs.ls(volume_path))
print(f"Found {len(csv_files)} file(s) — creating one bronze table per file:\n")

created_tables = ingest_volume_csvs(
    spark,
    dbutils,
    catalog=catalog,
    schema=schema,
    volume_path=volume_path,
    bronze_write_mode=bronze_write_mode,
    overwrite_schema=overwrite_schema,
)

for t in created_tables:
    print(f"  Table   : {t['table']}")
    print(f"  Source  : {t['source']}")
    print(f"  Columns : {', '.join(t['cols'])}")
    print(f"  Rows    : {t['rows']:,}")
    print()

print(f"{'─'*60}")
print(f"Bronze layer ready — {len(created_tables)} table(s) created in {catalog}.{schema}")
print(f"{'─'*60}")
for t in created_tables:
    print(f"  {t['table']:50s}  ({t['rows']:,} rows)")